In [ ]:
from collections import deque
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

# 1 - Default

In [ ]:
BASE_URL = "https://quotes.toscrape.com/"

In [ ]:
# All URLS
q = deque([BASE_URL])
visited = set()

i = 0

while q:
    url = q.popleft()
    i += 1
    if i >= 20:
        break
    visited.add(url)
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    for ref in soup.find_all("a", href=True):
        next_link = urljoin(response.url, ref["href"])
        if next_link not in visited:
            q.append(next_link)

In [ ]:
BASE_URL = "https://quotes.toscrape.com/page/{page_number}/"
page_number = 1
response = requests.get(BASE_URL.format(page_number=page_number))
soup = BeautifulSoup(response.text, "html.parser")
quotes = []

while True:
    for quote in soup.find_all("div", class_="quote"):
        quotes.append(quote.find("span", class_="text").text)

    next_btn = soup.find("li", class_="next")
    if not next_btn:  # Last page reached
        break

    page_number += 1
    soup = BeautifulSoup(requests.get(BASE_URL.format(page_number=page_number)).text, "html.parser")

In [ ]:
# CSS Selector

new_page = BASE_URL
quotes = []

while True:
    response = requests.get(new_page)
    soup = BeautifulSoup(response.text, "html.parser")

    for elem in soup.select(".quote"):
        quotes.append(elem.select(".text")[0].text)

    next_button = soup.select("ul li.next [href]")
    if next_button:
        new_page = urljoin(response.url, next_button[0]["href"])
    else:
        break

# 2 - Scroll

In [ ]:
BASE_URL = "https://quotes.toscrape.com/api/quotes?page={pn}"

In [ ]:
i = 1
quotes = []
while True:
    response = requests.get(BASE_URL.format(pn=i))

    if len(response.json()["quotes"]) == 0:
        break

    for quote in response.json()["quotes"]:
        quotes.append(quote["text"])

    i += 1

In [ ]:
len(quotes)

# 3 - JavaScript

In [ ]:
BASE_URL = "https://quotes.toscrape.com/js/"

In [ ]:
# uv run playwright install webkit -> lighter than chrome

In [ ]:
from playwright.async_api import async_playwright

quotes = []
async with async_playwright() as p:
    browser = await p.webkit.launch(headless=True)
    page = await browser.new_page()

    await page.goto(BASE_URL)

    while True:
        # Although this says await page.content() will return the current content
        # it will not wait for it to finish
        content = await page.content()
        page_quotes = await page.locator(".quote").all()
        for q in page_quotes:
            quotes.append(await q.text_content())

        next_button = await page.locator(".next a").all()
        if next_button:
            await next_button[0].click()
            # After you click you need to wait the content to finish loading
            await page.wait_for_load_state("domcontentloaded")
        else:
            break

len(quotes)

# 4 - JavaScript Delayed (10s)

In [ ]:
BASE_URL = "https://quotes.toscrape.com/js-delayed/"

In [ ]:
# uv run playwright install webkit -> lighter than chrome

In [ ]:
from playwright.async_api import async_playwright

quotes = []
async with async_playwright() as p:
    browser = await p.webkit.launch(headless=True)
    page = await browser.new_page()

    await page.goto(BASE_URL)

    while True:
        current_url = page.url
        print(current_url)

        # Waits until we can find at least one quote
        await page.wait_for_selector(".quote")

        # Although this says await page.content() will return the current content
        # it will not wait for it to finish
        content = await page.content()
        page_quotes = await page.locator(".quote").all()
        for q in page_quotes:
            quotes.append(await q.text_content())

        next_button = await page.locator(".next a").all()
        if next_button:
            await next_button[0].click()
            await page.wait_for_url(lambda url: url != current_url)
            # After you click you need to wait the content to finish loading
        else:
            break

len(quotes)

# 5 - Tableful

In [ ]:
new_page = "https://quotes.toscrape.com/tableful/page/{page_number}"
page_number = 1
quotes = []

while True:
    response = requests.get(new_page.format(page_number=page_number))
    print(response.url)
    soup = BeautifulSoup(response.text, "html.parser")

    quotes_css = soup.select('td[style="padding-top: 2em;"]')
    for quote in quotes_css:
        quotes.append(quote.text)  # Regex to remove the Author
    if not quotes_css:
        break

    page_number += 1

In [ ]:
len(quotes)

# 6 - Login

In [ ]:
import time

In [ ]:
BASE_URL = "https://quotes.toscrape.com/login"

In [ ]:
from playwright.async_api import async_playwright

In [ ]:
async with async_playwright() as p:
    browser = await p.webkit.launch(headless=False)
    page = await browser.new_page()

    await page.goto(BASE_URL)

    await page.fill("#username", "hello")
    time.sleep(4)
    await page.fill("#password", "password")
    time.sleep(4)
    await page.click("input.btn-primary")

# 7 - ViewState (AJAX)

In [ ]:
from playwright.async_api import Error as PlaywrightError
from playwright.async_api import async_playwright, expect

BASE_URL = "http://quotes.toscrape.com/search.aspx"


async def wait_for_detach(old_marker):
    """Wait until a previously captured element is gone (navigation or DOM swap)."""
    if old_marker is None:
        return
    try:
        await old_marker.wait_for_element_state("hidden")
    except PlaywrightError as e:
        if "not attached" not in str(e).lower():
            raise  # already detached => success


quotes = []

async with async_playwright() as p:
    browser = await p.webkit.launch(headless=True)
    page = await browser.new_page()

    await page.goto(BASE_URL)
    author_select = page.locator('select[name="author"]')
    tag_select = page.locator('select[name="tag"]')
    tag_options = tag_select.locator("option")

    authors = (await author_select.locator("option").all_text_contents())[1:]

    for author in authors:
        # Selecting an author navigates. The old tag dropdown may already be
        # populated (results page), so "options attached" is stale-satisfiable.
        # Capture a handle to the current tag <select> and wait for it to detach:
        old_tag_select = await tag_select.element_handle()
        await author_select.select_option(label=author)
        await wait_for_detach(old_tag_select)
        await expect(tag_options.nth(1)).to_be_attached()  # new page's tags populated

        tags = (await tag_options.all_text_contents())[1:]

        for tag in tags:
            # No author re-select: the results page keeps the author and the
            # tag dropdown. Selecting a tag does not navigate.
            await tag_select.select_option(label=tag)

            old_quote = None
            if await page.locator("div.quote").count() > 0:
                old_quote = await page.locator("div.quote").first.element_handle()

            await page.click(".btn")
            await wait_for_detach(old_quote)
            outcome = page.locator("div.quote").first.or_(page.get_by_text("No quotes found!"))
            await expect(outcome).to_be_visible()

            batch = await page.locator("div.quote span.content").all_text_contents()
            print(f"{author} / {tag}: {len(batch)} quotes")
            quotes.extend(batch)

    await browser.close()

print(f"\nTotal: {len(quotes)} quotes")

In [ ]:
len(quotes)

In [ ]:
len(set(quotes))